# KCIA v129 — Cloudflare / VPS / Mac / Colab 実機試験

まずWAN疎通と署名付きWebRTC接続を検証し、最後のセルで任意のPT試験を行います。Mac側は `README_WAN_PILOT_JA.md` を参照してください。

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()   # KCIA Cloudflare release ZIP を選択
assert len(uploaded)==1 and next(iter(uploaded)).endswith('.zip')
source = Path(next(iter(uploaded)))
root = Path('/content/kcia')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(source) as z:
    for entry in z.infolist():
        dest=(root/entry.filename).resolve()
        if not dest.is_relative_to(root.resolve()):
            raise ValueError('unsafe archive entry')
    z.extractall(root)
assert (root/'scripts/check_wan_p2p.py').exists()
print('KCIA sources extracted:',root)


In [ ]:
%cd /content/kcia
%pip -q install -r deploy/vps/requirements-transport.txt


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, 'scripts/wan_preflight.py', '--role', 'colab', '--require-deps'], check=True)


**設定**: Cloudflareで公開するシグナリング用ホスト名だけを指定します。クライアント用トークンは不要です。Colabに秘密鍵を持ち込まないでください。

In [ ]:
signal_host = input('Cloudflare signal hostname (signal.example.com): ').strip()
assert signal_host and '://' not in signal_host
signal_url = f'wss://{signal_host}/ws'


In [ ]:
import asyncio
from scripts.check_signaling_vps import probe
await probe(signal_url, timeout=20)


**鍵を生成**: 出力の `Public key (share this)` をMacに渡します。秘密鍵は Colab ランタイムの中だけにあります。

In [ ]:
from pathlib import Path
from patchlib.roomid import auto_identity
key = Path('/content/kcia/private/colab.pem')
peer, signer, public, verifier = auto_identity(key)
print('Colab peer ID:',peer)
print('Colab PUBLIC KEY (share with Mac):',public)


**Macの公開鍵・共有ルームIDを入力**: Mac 側でも `scripts/check_wan_p2p.py identity --key private/mac.pem` を実行し、公開鍵のみ入手してください。ランダムroom-idは両端で同じものを使用します。Macでも Colabでも、どちらを先に開始しても構いません。

In [ ]:
from patchlib.roomid import validate_room
mac_public = input('Mac public key (base64; NOT private PEM): ').strip()
room_id = validate_room(input('Shared random room ID (24-64 chars): ').strip())
turn_only = input('Force TURN relay candidates on BOTH hosts? (y/N): ').strip().lower() == 'y'
print('Start the corresponding Mac connect command now; run the final cell within ~180 sec.')


In [ ]:
from argparse import Namespace
from scripts.check_wan_p2p import connect
args = Namespace(key=key, ws=signal_url, token="", room_id=room_id,
                 remote_public_key=mac_public, turn_only=turn_only, timeout=180)
await connect(args)


## 任意: Colabで実PTを開始（上のハンドシェイクがPASSしてから）

Macで同一の `TRAIN_ROOM`, `HF_SHA`, モデル、rank、steps等を設定して `train-live` を起動してください。Colabでは CPU でなく CUDA を選べます。**Mac/Colab の状態ディレクトリ・秘密鍵は別々です。** Colabの学習依存ライブラリをインストールします。


In [ ]:
%pip -q install -r requirements-patches.txt


In [ ]:
import os, sys, subprocess, torch
from patchlib.roomid import validate_room
hf_sha = input('Pinned model revision (40 hex digits): ').strip()
assert len(hf_sha)==40 and all(x in '0123456789abcdef' for x in hf_sha.lower())
train_room = validate_room(input('TRAIN_ROOM (same as Mac; new random room): ').strip())
assert input('Run PT on Colab after Mac PT has started? type RUN: ').strip() == 'RUN'
state_dir = Path('/content/kcia/wan_state/colab')
state_dir.mkdir(parents=True,exist_ok=True)
cmd = [sys.executable, '-m','patchlib.cli','train-live',
 '--ws',signal_url,'--room-id',train_room,
 '--model','HuggingFaceTB/SmolLM2-135M','--revision',hf_sha,
 '--auto-state','--state-base',str(state_dir/'genesis.fstate'),
 '--text-file','examples/wan_sample.txt',
 '--signing-key',str(key),
 '--kind','loha','--rank','32','--local-loha-rank','auto','--reinit-every','20',
 '--device','cuda' if torch.cuda.is_available() else 'cpu',
 '--batch-size','1','--max-length','64','--local-steps','4',
 '--journal',str(state_dir/'events.sqlite3'),'--rounds','12']
env = os.environ.copy()
print('Running public-signaling PT-only test. No client token is required.')
subprocess.run(cmd, cwd=root, env=env, check=True)
